## 4章 多言語の固有認識表現

##### 4.1 データセット
+ XTREME : Cross-lingual TRansfer Evaluation of Multilingal Encoders
+ PAN-Xを探す

In [1]:
from huggingface_hub import notebook_login
# notebook_login()

In [2]:
from pprint import pprint
from datasets import get_dataset_config_names

In [3]:
xtreme_subsets = get_dataset_config_names("xtreme")
pprint(xtreme_subsets)
print(f"XTREME has {len(xtreme_subsets)} configurations")

['MLQA.ar.ar',
 'MLQA.ar.de',
 'MLQA.ar.en',
 'MLQA.ar.es',
 'MLQA.ar.hi',
 'MLQA.ar.vi',
 'MLQA.ar.zh',
 'MLQA.de.ar',
 'MLQA.de.de',
 'MLQA.de.en',
 'MLQA.de.es',
 'MLQA.de.hi',
 'MLQA.de.vi',
 'MLQA.de.zh',
 'MLQA.en.ar',
 'MLQA.en.de',
 'MLQA.en.en',
 'MLQA.en.es',
 'MLQA.en.hi',
 'MLQA.en.vi',
 'MLQA.en.zh',
 'MLQA.es.ar',
 'MLQA.es.de',
 'MLQA.es.en',
 'MLQA.es.es',
 'MLQA.es.hi',
 'MLQA.es.vi',
 'MLQA.es.zh',
 'MLQA.hi.ar',
 'MLQA.hi.de',
 'MLQA.hi.en',
 'MLQA.hi.es',
 'MLQA.hi.hi',
 'MLQA.hi.vi',
 'MLQA.hi.zh',
 'MLQA.vi.ar',
 'MLQA.vi.de',
 'MLQA.vi.en',
 'MLQA.vi.es',
 'MLQA.vi.hi',
 'MLQA.vi.vi',
 'MLQA.vi.zh',
 'MLQA.zh.ar',
 'MLQA.zh.de',
 'MLQA.zh.en',
 'MLQA.zh.es',
 'MLQA.zh.hi',
 'MLQA.zh.vi',
 'MLQA.zh.zh',
 'PAN-X.af',
 'PAN-X.ar',
 'PAN-X.bg',
 'PAN-X.bn',
 'PAN-X.de',
 'PAN-X.el',
 'PAN-X.en',
 'PAN-X.es',
 'PAN-X.et',
 'PAN-X.eu',
 'PAN-X.fa',
 'PAN-X.fi',
 'PAN-X.fr',
 'PAN-X.he',
 'PAN-X.hi',
 'PAN-X.hu',
 'PAN-X.id',
 'PAN-X.it',
 'PAN-X.ja',
 'PAN-X.jv',
 'PAN

In [4]:
panx_subsets = [s for s in xtreme_subsets if s.startswith("PAN")]
panx_subsets[:5]

['PAN-X.af', 'PAN-X.ar', 'PAN-X.bg', 'PAN-X.bn', 'PAN-X.de']

In [5]:
# ドイツ語のコーパスを取得
from datasets import load_dataset

tmp_ds = load_dataset("xtreme", name="PAN-X.de")
print(tmp_ds)
for it in tmp_ds:
    print(it)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
})
train
validation
test


In [6]:
# 各言語(ドイツ語, フランス語, イタリア語, 英語)のコーパスから
# 特定の割合でサンプリングしてデータセットを作る
from collections import defaultdict
from datasets import DatasetDict

langs = ["de", "fr", "it", "en"]
fracs = [0.629, 0.229, 0.084, 0.059]

# キーが存在しなければ, DatasetDictを返す
panx_ch = defaultdict(DatasetDict)
for lang, frac in zip(langs, fracs):
    # 単語コーパスをロード
    ds = load_dataset("xtreme", name=f"PAN-X.{lang}")
    # 各分割をシャッフルし、話者の割合に応じてダウンサンプリング
    for split in ds: # train, validation, test
        print(f"langs: {lang}, split: {split}")
        panx_ch[lang][split] = (
            ds[split]
            .shuffle(seed=0)
            .select(range(int(frac * ds[split].num_rows)))
        )


langs: de, split: train
langs: de, split: validation
langs: de, split: test
langs: fr, split: train
langs: fr, split: validation
langs: fr, split: test
langs: it, split: train
langs: it, split: validation
langs: it, split: test
langs: en, split: train
langs: en, split: validation
langs: en, split: test


In [7]:
print(panx_ch)

defaultdict(<class 'datasets.dataset_dict.DatasetDict'>, {'de': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
}), 'fr': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 4580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 2290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 2290
    })
}), 'it': DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 1680
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 840
    })
    test: Dataset({
        features: ['token

In [8]:
# 各データセットのnum_row属性にアクセスして学習データに含まれる言語ごとの事例を確認
import pandas as pd

pd.DataFrame({lang: [panx_ch[lang]["train"].num_rows] for lang in langs},
             index=["Number of training examples"])

,de,fr,it,en
Number of training examples,12580,4580,1680,1180


In [9]:
# ドイツ語コーパスに含まれる一つの事例を検証
element = panx_ch['de']['train'][0]
for key, value in element.items():
    print(f"{key}: {value}")

tokens: ['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags: [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs: ['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


In [10]:
for key, value in panx_ch["de"]["train"].features.items():
    print(f"{key}: {value}")

tokens: List(Value('string'))
ner_tags: List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs: List(Value('string'))


In [11]:
panx_ch["de"]["train"]

Dataset({
    features: ['tokens', 'ner_tags', 'langs'],
    num_rows: 12580
})

In [12]:
ner_tags = panx_ch["de"]["train"].features['ner_tags']
ner_tags

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))

In [13]:
ner_tags.feature

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])

In [14]:
ner_tags.feature.__dict__

{'names': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'],
 'id': None,
 'num_classes': 7,
 'names_file': None,
 '_int2str': ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'],
 '_str2int': {'O': 0,
  'B-PER': 1,
  'I-PER': 2,
  'B-ORG': 3,
  'I-ORG': 4,
  'B-LOC': 5,
  'I-LOC': 6}}

In [15]:
tags = panx_ch["de"]["train"].features["ner_tags"].feature
print("tag: ", tags)

def create_tag_names(batch):
    # tagsは int <-> str の変換ルール.
    return {"ner_tags_str": [tags.int2str(idx) for idx in batch["ner_tags"]]}

tag:  ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [16]:
panx_ch["de"]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 6290
    })
})

In [17]:
panx_de = panx_ch["de"].map(create_tag_names)

In [18]:
panx_de

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'ner_tags_str'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'ner_tags_str'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'ner_tags_str'],
        num_rows: 6290
    })
})

In [19]:
de_example = panx_de["train"][0] # trainデータの一つ目
pd.DataFrame([de_example["tokens"], de_example["ner_tags_str"]], ["Tokens", "Tags"])

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [20]:
# タグに異常な不均衡がないことを簡単に確認するために
# 各分割における各固有表現の頻度を計算する
from collections import Counter

split2freqs = defaultdict(Counter)
for split, dataset in panx_de.items():
    for row in dataset["ner_tags_str"]:
        for tag in row:
            if tag.startswith("B"):
                tag_type = tag.split("-")[1]
                split2freqs[split][tag_type] += 1

pd.DataFrame.from_dict(split2freqs, orient="index")

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


#### 4.2 多言語Transformer

SentencePiece VS WordPiece
+ XLM-R系モデルは生のテキストを直接トークン化するSentencePieceを使う.
+ 100言語の生テキストで学習させたSentencePieceを使う.
+ BERTはWordPieceを使っている?
+

In [21]:
# BERTとXML-Rをロード
from transformers import AutoTokenizer

bert_model_name = "bert-base-cased"
xlmr_model_name = "xlm-roberta-base"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_model_name)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [22]:
# テキストをエンコードすることで、各モデルが事前学習時に使用した特殊トークンを取得できる
text = "Jack Sparrow loves New York!"
bert_tokens = bert_tokenizer(text).tokens()
xlmr_tokens = xlmr_tokenizer(text).tokens()
print(bert_tokens) # [CLS], ##, [SEP] ... WordPiece
print(xlmr_tokens) # <s>, _***, </s> ... SentencePiece

['[CLS]', 'Jack', 'Spa', '##rrow', 'loves', 'New', 'York', '!', '[SEP]']
['<s>', '▁Jack', '▁Spar', 'row', '▁love', 's', '▁New', '▁York', '!', '</s>']


#### トークナイザーのパイプライン
1. 正規化 : 空白の除去, アクセント付き文字の除去, Unicode正規化(NFC,NFD,NFKC,NFKDを1つに統一), 小文字化
2. 事前トークン化 : BPE(Byte-Pair Encoding)やUnigramアルゴリズム, WordPieceなどでテキストをトークン列に分割. or 言語圏特有のライブラリの使用.
3. トークナイザーモデル : (学習済み)モデルでトークン列をサブワードに分割する. (サブワード分割モデルの適用)
4. 後処理 : 特殊トークンの追加など

In [23]:
# SentencePieceトークナイザー
"".join(xlmr_tokens) # _ : Lower One Quarter Block

'<s>▁Jack▁Sparrow▁loves▁New▁York!</s>'

In [24]:
"".join(xlmr_tokens).replace(u"\u2581", " ")

'<s> Jack Sparrow loves New York!</s>'

#### 固有表現認識用のTransformer
+ タスク専用のヘッドを自作する
+ HuggingFaceに既に存在するヘッドを利用する. この場合は、XLMRobertaForTokenClassificationクラス

In [25]:
import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel

class XLMRobertaForTokenClassification(RobertaPreTrainedModel):
    config_class = XLMRobertaConfig

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # モデルボディのロード
        self.roberta = RobertaModel(config, add_pooling_layer=False) # add_pooling_layer=Falseで[CLS]以外の全ての出力に関連する隠れ層の重みを取り出す
        # トークン分類ヘッドの用意
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        # 重みのロードと初期化
        self.init_weights() # RobertaPreTrainedModel

    def forward(self,
                input_ids=None,
                attention_mask=None,
                token_type_ids=None,
                labels=None,
                **kwargs):
        # モデルボディを使ってエンコーダの表現を取得
        outputs = self.roberta(input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids, **kwargs)

        # 分類器をエンコーダ表現に適用(ヘッド)
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)

        # 損失の計算(forwardで予測誤差を一応計算)
        loss = None
        if labels is not None:
            # もしアテンションマスクがあれば、マスクしていない部分のみのトークンで損失を計算すべき
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        # モデルの出力オブジェクトを返す
        return TokenClassifierOutput(loss=loss, logits=logits,
                                     hidden_states=outputs.hidden_states,
                                     attentions=outputs.attentions)




カスタムモデルのロード

In [26]:
index2tag = {idx: tag for idx, tag in enumerate(tags.names)}
tag2index = {tag: idx for idx, tag in enumerate(tags.names)}

In [27]:
from transformers import AutoConfig

xlmr_config = AutoConfig.from_pretrained(xlmr_model_name,
                                         num_labels=tags.num_classes,
                                         index2label=index2tag, label2id=tag2index)

In [28]:
import torch

print(torch.cuda.device_count())
print(torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: %s" % device)
# xlmr_config = xlmr_config.to(device)
xlmr_model = (XLMRobertaForTokenClassification
              .from_pretrained(xlmr_model_name, config=xlmr_config)
              .to(device))


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


1
True
device: cuda


In [29]:
# トークナイザーとモデルの動作チェック
input_ids = xlmr_tokenizer.encode(text, return_tensors="pt")
pd.DataFrame([xlmr_tokens, input_ids[0].numpy()], index=["Tokens", "Input IDs"])

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>
Input IDs,0,21763,37456,15555,5161,7,2356,5753,38,2


In [30]:
outputs = xlmr_model(input_ids.to(device)).logits
predictions = torch.argmax(outputs, dim=-1)
print(f"Number of tokens in sequence: {len(xlmr_tokens)}")
print(f"Shape of outputs: {outputs.shape}") # (Batch,Token,Class)

Number of tokens in sequence: 10
Shape of outputs: torch.Size([1, 10, 7])


In [31]:
preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
pd.DataFrame([xlmr_tokens, preds], index=["Tokens", "Tags"]) # ファインチューニングしていないので適当な値が出る

,0,1,2,3,4,5,6,7,8,9
Tokens,<s>,▁Jack,▁Spar,row,▁love,s,▁New,▁York,!,</s>
Tags,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC,I-LOC


In [32]:
# ファインチューニングのためのヘルパー関数
def tag_text(text, tags, model, tokenizer):
    # 特殊な文字列を含むトークンを取得
    tokens = tokenizer(text).tokens()
    # 系列をIDにエンコード
    input_ids = xlmr_tokenizer(text, return_tensors="pt").input_ids.to(device)
    # 7つのクラス分布にわたる予測を得る
    outputs = model(input_ids)[0]
    # argmaxを使い、トークンごとに尤も可能性の高いクラスを取得
    predictions = torch.argmax(outputs, dim=2)
    # DataFrameへ変換
    preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
    return pd.DataFrame([tokens, preds], index=["Tokens", "Tags"])

#### データセット全体をトークン化して, XLM-Rモデルに渡してファインチューニング

In [33]:
# 単語とNERラベル
pprint(de_example)
words, labels = de_example["tokens"], de_example["ner_tags"]
print(words)
print(labels)

{'langs': ['de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de',
           'de'],
 'ner_tags': [0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0],
 'ner_tags_str': ['O',
                  'O',
                  'O',
                  'O',
                  'B-LOC',
                  'I-LOC',
                  'O',
                  'O',
                  'B-LOC',
                  'B-LOC',
                  'I-LOC',
                  'O'],
 'tokens': ['2.000',
            'Einwohnern',
            'an',
            'der',
            'Danziger',
            'Bucht',
            'in',
            'der',
            'polnischen',
            'Woiwodschaft',
            'Pommern',
            '.']}
['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
[0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]


In [34]:
# 単語をトークン化
tokenized_input = xlmr_tokenizer(de_example["tokens"], is_split_into_words=True)
tokens = xlmr_tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
pd.DataFrame([tokens], index=["Tokens"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>


In [35]:
# トークンをサブワードに分割した場合のラベル対応は先頭のみとする慣習に従うとする
# e.g. Einwohnern -> _Einwohner と n -> _Einwohnerのみに「B-LOC」ラベルを付与する
# word_ids()でマスキングできる
word_ids = tokenized_input.word_ids()
pd.DataFrame([tokens, word_ids], index=["Tokens", "Word IDs"])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None


In [36]:
# 学習時に利用しない特殊なトークンや学習時にマスクしたいサブワードのラベルを-100とする
previous_word_idx = None
label_ids = []

for word_idx in word_ids: # word_ids: サブワードの塊
    if word_idx is None or word_idx == previous_word_idx:
        label_ids.append(-100)
    elif word_idx != previous_word_idx:
        label_ids.append(labels[word_idx])
    previous_word_idx = word_idx

labels = [index2tag[l] if l != -100 else "IGN" for l in label_ids]
index = ["Tokens", "Word IDs", "Label IDs", "Labels"]

pd.DataFrame([tokens, word_ids, label_ids, labels], index=index)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None
Label IDs,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
Labels,IGN,O,O,IGN,O,O,B-LOC,IGN,IGN,I-LOC,...,B-LOC,IGN,IGN,IGN,I-LOC,IGN,IGN,O,IGN,IGN


In [37]:
print(panx_de["train"])
trn_examples = panx_de["train"]
trn_examples
trn_examples['tokens']
tokenized_inputs = xlmr_tokenizer(list(trn_examples['tokens']), truncation=True, is_split_into_words=True)
tokenized_inputs[:2]

Dataset({
    features: ['tokens', 'ner_tags', 'langs', 'ner_tags_str'],
    num_rows: 12580
})


[Encoding(num_tokens=25, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=19, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [38]:
# データセット全体に上記セルを適用する関数
def tokenize_and_align_labels(examples): # examples: 'train', 'validation', 'test'
    tokenized_inputs = xlmr_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for idx, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=idx)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None or word_idx == previous_word_idx:
                label_ids.append(-100) # torch.nn.CrossEntropyLossにignore_index属性があり、これが-100
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [39]:
# 反復処理用関数
def encode_panx_dataset(corpus):
    return corpus.map(tokenize_and_align_labels, batched=True,
                      remove_columns=["langs", "ner_tags", "tokens"])


In [40]:
# DatasetDictに適用して分割ごとにエンコードされたDatasetを得ることができる
panx_de_encoded = encode_panx_dataset(panx_ch["de"])
panx_de_encoded

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6290
    })
})

In [41]:
trn_labels_0 = panx_de_encoded["train"]["labels"][0]
trn_input_ids_0 = panx_de_encoded["train"]["input_ids"][0]
trn_attention_mask_0 = panx_de_encoded["train"]["attention_mask"][0]

pd.DataFrame([trn_labels_0, trn_attention_mask_0, trn_input_ids_0], index=["label", "amask", 'input_id'])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
label,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
amask,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
input_id,0,70101,176581,19,142,122,2290,708,1505,18363,...,13787,14,15263,18917,663,6947,19,6,5,2


In [42]:
panx_de_encoded

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12580
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6290
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6290
    })
})

### 性能指標

In [43]:
from seqeval.metrics import classification_report

y_true = [
    ["0", "0", "0", "B-MISC", "I-MISC", "I-MISC", "0"],
    ["B-PER", "I-PER", "0"]
]
y_pred = [
    ["0", "0", "B-MISC", "I-MISC", "I-MISC", "I-MISC", "0"],
    ["B-PER", "I-PER", "0"]
]
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

        MISC       0.00      0.00      0.00         1
         PER       1.00      1.00      1.00         1
           _       0.00      0.00      0.00         1

   micro avg       0.33      0.33      0.33         3
   macro avg       0.33      0.33      0.33         3
weighted avg       0.33      0.33      0.33         3



C:\Users\inoue\Documents\MyGithub\Book_Transformers\.venv\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [44]:
# 学習中にseqevalを適用するための関数
import numpy as np

def align_predictions(predictions, label_ids):
    preds = np.argmax(predictions, axis=2)
    batch_size, seq_len = preds.shape
    labels_list, preds_list = [], []

    for batch_idx in range(batch_size):
        example_labels, example_preds = [], []
        for seq_idx in range(seq_len):
            # ラベルIDが-100の場合は無視
            if label_ids[batch_idx, seq_idx] != -100:
                example_labels.append(index2tag[label_ids[batch_idx][seq_idx]])
                example_preds.append(index2tag[preds[batch_idx][seq_idx]])

            labels_list.append(example_labels)
            preds_list.append(example_preds)

    return preds_list, labels_list


## XLM-RoBERTaのファインチューニング

+ PAN-Xのドイツ語サブセットでベースモデルをファインチューニング
+ フランス語、イタリア語、英語でゼロショット言語観転移性能を評価する

In [45]:
from transformers import TrainingArguments

num_epochs = 3
batch_size = 24
logging_steps = len(panx_de_encoded["train"]) // batch_size
model_name = f"{xlmr_model_name}-finetuned-panx-de"
training_args = TrainingArguments(
    output_dir=model_name,
    log_level="error",
    num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="epoch",
    save_steps=1e6,
    weight_decay=0.01,
    disable_tqdm=False,
    logging_steps=logging_steps,
    push_to_hub=True
)

In [46]:
# Huffing Face Hubにログインしていることを確認
from huggingface_hub import notebook_login
# notebook_login()

In [47]:
# F1スコアの計算
from seqeval.metrics import f1_score

def compute_metrics(eval_pred):
    y_pred, y_true = align_predictions(eval_pred.predictions, eval_pred.label_ids)
    return {"f1": f1_score(y_true, y_pred)}

In [48]:
# データコレータ(data collator)を定義して各入力系列をバッチで最大の系列長になるようにパディングする
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)

In [49]:
def model_init():
    return (XLMRobertaForTokenClassification
            .from_pretrained(xlmr_model_name, config=xlmr_config)
            .to(device))

In [50]:
from transformers import Trainer

trainer = Trainer(model_init=model_init,
                  args=training_args,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  train_dataset=panx_de_encoded["train"],
                  eval_dataset=panx_de_encoded["validation"],
                  tokenizer=xlmr_tokenizer,
                  )

In [51]:
# 学習ループを実行して、最終的なモデルをHubにプッシュする
trainer.train()
trainer.push_to_hub(commit_message="Training completed! by tiny-tank!")

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### エラー分析

In [52]:
# 検証データセットに適用するメソッド
# 系列サンプル毎の損失値を計算する

from torch.nn.functional import cross_entropy

def forward_pass_with_label(batch):
    # リストの辞書をデータコレータに適した辞書のリストに変換
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]

    # 入力トラベルをパディングし、すべてのテンソルをデバイス上に載せる
    batch = data_collator(features)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.no_grad():
        # データをモデルに渡す
        output = trainer.model(input_ids, attention_mask)
        # logit.size: [batch_size, sequence_length, classes]
        # ロジット値が最大のクラスを予測する
        predicted_label = torch.argmax(output.logits, axis=-1).cpu().numpy()

    # viewを使ってバッチの次元をFlattenした後、トークンごとの損失を計算
    loss = cross_entropy(output.logits.view(-1, 7), labels.view(-1), reduction="none")

    # バッチ次元をUnflattenし、numpy配列に変換
    loss = loss.view(len(input_ids), -1).cpu().numpy()

    return {"loss": loss, "predicted_label": predicted_label}




In [53]:
# map()を使って検証データセット全体に適用して、すべてのデータをDataFrameにロードして分析
valid_set = panx_de_encoded["validation"]
valid_set = valid_set.map(forward_pass_with_label, batched=True, batch_size=32)
df = valid_set.to_pandas()

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]